In [14]:
import os
os.chdir("/kaggle/working")
!rm -rf dr_grader_diss

In [15]:

import subprocess, sys

# clone repo into working dir
subprocess.run(["git", "clone", "https://github.com/simransarah/dr_grader_diss.git"])

# timm is not pre-installed on Kaggle, everything else is
subprocess.run([sys.executable, "-m", "pip", "install", "timm", "-q"], check=True)

Cloning into 'dr_grader_diss'...


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', 'timm', '-q'], returncode=0)

In [17]:
config_code = '''import os
import torch

class SegmentationConfig:
    TARGET_LESION = "ex"

    root_dir = os.path.join("/kaggle", "input", "idriddata", "data", "IDRiD", "A. Segmentation")

    train_images_dir = os.path.join(root_dir, "1. Original Images", "a. Training Set")
    train_masks_dir  = os.path.join(root_dir, "2. All Segmentation Groundtruths", "a. Training Set")

    val_images_dir = os.path.join(root_dir, "1. Original Images", "b. Testing Set")
    val_masks_dir  = os.path.join(root_dir, "2. All Segmentation Groundtruths", "b. Testing Set")

    checkpoint_dir = "/kaggle/working"

    backbone = "efficientnet-b3"
    pretrained = True
    patch_size = (512, 512)
    in_channels = 3
    out_channels = 1
    batch_size = 4
    virtual_batch_size = 16
    learning_rate = 1e-4
    num_epochs = 150
    num_classes = 3
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    clahe_probability = 1
    grid_distortion_probability = 0.5
'''

with open("/kaggle/working/dr_grader_diss/segmentation/config.py", "w") as f:
    f.write(config_code)

print("Config written.")

Config written.


In [18]:
import os

paths_to_check = [
    "/kaggle/input/idriddata/data/IDRiD/A. Segmentation/1. Original Images/a. Training Set",
    "/kaggle/input/idriddata/data/IDRiD/A. Segmentation/1. Original Images/b. Testing Set",
    "/kaggle/input/idriddata/data/IDRiD/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set",
    "/kaggle/input/idriddata/data/IDRiD/A. Segmentation/2. All Segmentation Groundtruths/b. Testing Set",
]

for p in paths_to_check:
    exists = os.path.exists(p)
    count  = len(os.listdir(p)) if exists else 0
    print(f"{'OK' if exists else 'MISSING'} ({count} files)  {p}")

OK (54 files)  /kaggle/input/idriddata/data/IDRiD/A. Segmentation/1. Original Images/a. Training Set
OK (27 files)  /kaggle/input/idriddata/data/IDRiD/A. Segmentation/1. Original Images/b. Testing Set
OK (5 files)  /kaggle/input/idriddata/data/IDRiD/A. Segmentation/2. All Segmentation Groundtruths/a. Training Set
OK (5 files)  /kaggle/input/idriddata/data/IDRiD/A. Segmentation/2. All Segmentation Groundtruths/b. Testing Set


In [20]:
!pip install monai segmentation-models-pytorch albumentations shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 15.3 MB/s eta 0:00:00


In [22]:
import sys
import os

# Add your project root to the system path
project_root = '/kaggle/working/dr_grader_diss'
if project_root not in sys.path:
    sys.path.append(project_root)

# Also set it as an environment variable for any subprocesses
os.environ['PYTHONPATH'] = project_root

print("Python path updated. Ready to run!")

Python path updated. Ready to run!


In [ ]:
os.chdir("/kaggle/working/dr_grader_diss")

%run run_stage1.py


Starting Stage 1 Training for: EX...


2026-03-23 17:47:42.939119: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774288062.960168     264 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774288062.966788     264 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774288062.983930     264 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774288062.983956     264 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774288062.983959     264 computation_placer.cc:177] computation placer alr

Starting Specialist Training for: EX using efficientnet-b3...


Epoch 1:   0%|          | 0/14 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/monai/losses/dice.py:175: UserWarning: single channel prediction, `include_background=False` ignored.
  warnings.warn("single channel prediction, `include_background=False` ignored.")
/usr/local/lib/python3.12/dist-packages/monai/losses/focal_loss.py:147: UserWarning: single channel prediction, `include_background=False` ignored.
  warnings.warn("single channel prediction, `include_background=False` ignored.")
Epoch 1: 100%|██████████| 14/14 [00:29<00:00,  2.14s/it, loss=1.1849]
/usr/local/lib/python3.12/dist-packages/monai/inferers/utils.py:226: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_vari

Epoch 1 Results: EX Dice: 0.0127
>>> New Best EX Model Saved :)


...

Epoch 128: 100%|██████████| 14/14 [00:24<00:00,  1.75s/it, loss=0.6490]


Epoch 128 Results: HE Dice: 0.2498


Epoch 129: 100%|██████████| 14/14 [00:24<00:00,  1.74s/it, loss=0.5349]


Epoch 129 Results: HE Dice: 0.2512


Epoch 130:   0%|          | 0/14 [00:00<?, ?it/s]

In [ ]:
# 1. Move into the repo folder
%cd /kaggle/working/dr_grader_diss

# 2. Run the module (now it will find 'segmentation')
!python -m segmentation.train_monai

/kaggle/working/dr_grader_diss
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotate